<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《Build a Large Language Model From Scratch》</a> 一书的补充代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第 2 章：处理文本数据

本 Notebook 使用的软件包：

torch 是 PyTorch 的核心包。这个项目用它来实现和训练神经网络，包括张量运算、embedding 层、attention、Transformer/GPT 模型、反向传播、优化器、数据集和 DataLoader 等。

tiktoken 是 OpenAI 开源的 tokenizer 包。这个项目在第 2 章用它演示 GPT-2 风格的 BPE 分词，把文本转换成 token ID，也可以把 token ID 解码回文本。后续训练 GPT 时，模型实际吃进去的不是原始字符串，而是这些 token ID。

In [13]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.12.0
tiktoken version: 0.13.0


- 本章介绍数据准备和采样，让输入数据可以“准备好”供 LLM 使用

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/01.webp?timestamp=1" width="1000px">

&nbsp;
## 2.1 理解词嵌入

- 本节没有代码

- 嵌入有很多形式；本书重点关注文本嵌入

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/02.webp" width="1000px">

- LLM 使用高维空间中的嵌入（也就是成千上万个维度）
- 由于我们无法可视化这种高维空间（人类通常以 1 维、2 维或 3 维来思考），下图用二维嵌入空间做说明

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/03.webp" width="600px">

&nbsp;
## 2.2 文本分词

- 本节会对文本进行分词，也就是把文本拆成更小的单元，例如单个单词和标点符号

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/04.webp" width="600px">

- 加载我们要处理的原始文本
- [Edith Wharton 的《The Verdict》](https://en.wikisource.org/wiki/The_Verdict) 是一篇公版短篇小说

In [14]:
import os #Python 标准库，用来和操作系统交互。这个 Notebook 里主要用它检查本地文件是否已经存在
import requests #第三方库，用来发送 HTTP 请求。我们用它来下载文本文件

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)


# 本书原先使用下面的代码
# 不过，urllib 使用较旧的协议设置，
# 可能会给部分使用 VPN 的读者带来问题。
# 上面的 `requests` 版本在这方面
# 更加稳健。

"""
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)
"""

'\nimport os\nimport urllib.request\n\nif not os.path.exists("the-verdict.txt"):\n    url = ("https://raw.githubusercontent.com/rasbt/"\n           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"\n           "the-verdict.txt")\n    file_path = "the-verdict.txt"\n    urllib.request.urlretrieve(url, file_path)\n'

<br>

---

<br>

#### SSL 证书错误排查

- 有些读者反馈，在 VSCode 或 Jupyter 中运行 `urllib.request.urlretrieve` 时会看到 ssl.SSLCertVerificationError：`SSL: CERTIFICATE_VERIFY_FAILED`。
- 这通常表示 Python 的证书包已经过期。


**修复方法**

- 使用 Python ≥ 3.9；可以执行下面的代码检查 Python 版本：
```python
import sys
print(sys.__version__)
```
- 升级证书包：
  - pip：`pip install --upgrade certifi`
  - uv：`uv pip install --upgrade certifi`
- 升级后重启 Jupyter kernel。
- 如果执行前一个代码单元时仍然遇到 `ssl.SSLCertVerificationError`，请参阅 GitHub 上的[更多讨论](https://github.com/rasbt/LLMs-from-scratch/pull/403)。

<br>

---

<br>

In [15]:
# 现在我们已经有了文本文件，来看看它的内容吧。
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- 目标是为 LLM 对这段文本进行分词和嵌入
- 我们先基于一小段简单示例文本开发一个简单 tokenizer，之后再把它应用到上面的文本
- 下面这个正则表达式会按空白字符切分文本
  

re.split(pattern, text) 的意思是：按照 pattern 匹配到的位置切分 text。

In [16]:
import re #Python 标准库，用来处理正则表达式。我们用它来分割文本

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
#使用正则表达式 `r'(\s)'` 来分割文本。这个表达式匹配任何空白字符（如空格、制表符等）
# 并且括号 `()` 表示我们想要保留这些分隔符在结果中。

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


- 我们不只想按空白字符切分，还想按逗号和句号切分，所以需要相应地修改正则表达式

In [17]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


- 可以看到，这会产生空字符串；我们把它们移除

In [18]:
# 去掉每个条目两端的空白字符，然后过滤掉所有空字符串。
result = [item for item in result if item.strip()]# 过滤掉空字符串
                #item        # 放进新列表的值
                #for item in result   # 从 result 里一个个取值
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


- 这样已经不错了，不过我们还要处理句号、问号等其他标点符号

In [19]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


- 现在效果已经不错，我们可以把这种分词方式应用到原始文本上

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/05.webp" width="500px">

In [20]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


- 计算 token 总数

In [21]:
print(len(preprocessed))

4690


&nbsp;
## 2.3 将 token 转换为 token ID

- 接下来，把文本 token 转换为 token ID，后续就可以通过嵌入层处理它们

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/06.webp" width="1000px">

- 基于这些 token，现在可以构建一个由所有唯一 token 组成的词表

In [22]:
all_words = sorted(set(preprocessed))#set() 函数用于创建一个无序不重复元素集。我们用它来获取文本中所有唯一的词（token）
vocab_size = len(all_words)
print(vocab_size)

1130


In [23]:
vocab = {token:integer for integer,token in enumerate(all_words)}
# 构建一个字典，将每个唯一的词（token）映射到一个整数索引
#token: integer，表示字典里的键值对
#enumerate(all_words)，为 all_words 列表中的每个元素生成一个索引和值的对。索引从 0 开始，值是列表中的元素。

- 下面是这个词表中的前 50 个条目：

In [24]:
for i, item in enumerate(vocab.items()):
#items() 方法返回一个包含字典键值对的视图对象。每个键值对以元组的形式表示，格式为 (key, value)。
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


- 下面用一个小词表演示短文本的分词过程：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/07.webp?123" width="1000px">

- 现在把这些步骤组合成一个 tokenizer 类

In [25]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # 替换指定标点符号前面的空格
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

- `encode` 函数把文本转换为 token ID
- `decode` 函数把 token ID 转回文本

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/08.webp?123" width="1000px">

- 可以使用 tokenizer 将文本编码（也就是分词）为整数
- 之后这些整数可以作为 LLM 的输入被嵌入

In [26]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


- 可以把整数解码回文本

In [27]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [28]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

&nbsp;
## 2.4 添加特殊上下文 token

- 添加一些“特殊” token 很有用，例如表示未知词或文本结尾

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/09.webp?123" width="1000px">

- 一些 tokenizer 会使用特殊 token，为 LLM 提供额外上下文
- 其中一些特殊 token 包括
  - `[BOS]`（beginning of sequence）标记文本开头
  - `[EOS]`（end of sequence）标记文本结束的位置（通常用于拼接多段无关文本，例如两篇不同的维基百科文章或两本不同的书等）
  - `[PAD]`（padding）用于 batch size 大于 1 的 LLM 训练（一个 batch 中可能包含长度不同的多段文本；通过 padding token 把较短文本补到最长长度，使所有文本长度一致）
- `[UNK]` 用于表示词表中没有包含的词

- 注意，GPT-2 并不需要上面这些 token；为了降低复杂度，它只使用 `<|endoftext|>` token
- `<|endoftext|>` 与上面提到的 `[EOS]` token 类似
- GPT 也会把 `<|endoftext|>` 用作 padding（因为用 batch 输入训练时通常会使用 mask，模型本来也不会关注被 padding 的 token，所以这些 token 具体是什么并不重要）
- GPT-2 不使用 `<UNK>` token 来表示词表外词；相反，GPT-2 使用字节对编码（BPE）tokenizer，把单词拆成子词单元，后面的章节会讨论这一点


- 我们在两段独立文本之间使用 `<|endoftext|>` token：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/10.webp" width="1000px">

- 看看对下面这段文本分词会发生什么：

In [29]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

KeyError: 'Hello'

- 上面的代码会报错，因为单词 "Hello" 不在词表中
- 为了处理这种情况，可以向词表中添加 `"<|unk|>"` 这样的特殊 token，用它表示未知词
- 既然已经在扩展词表，我们再添加另一个名为 `"<|endoftext|>"` 的 token；它在 GPT-2 训练中用于表示文本结束（也用于拼接文本之间，例如训练数据集由多篇文章、书籍等组成时）

In [30]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [31]:
len(vocab.items())

1132

In [32]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


- 还需要相应地调整 tokenizer，让它知道何时以及如何使用新的 `<unk>` token

In [33]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]# 对于不在词汇表中的词，用 "<|unk|>" 替换

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # 替换指定标点符号前面的空格
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

试着用修改后的 tokenizer 对文本分词：

In [34]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [35]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [36]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

&nbsp;
## 2.5 字节对编码（BytePair encoding）

- GPT-2 使用字节对编码（BPE）作为 tokenizer
- 它允许模型把预定义词表中不存在的单词拆成更小的子词单元，甚至拆成单个字符，从而处理词表外词
- 例如，如果 GPT-2 的词表中没有 "unfamiliarword" 这个词，它可能会根据训练好的 BPE 合并规则，将其分成 ["unfam", "iliar", "word"]，或其他子词组合
- 原始 BPE tokenizer 见这里：[https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- 本章使用 OpenAI 开源 [tiktoken](https://github.com/openai/tiktoken) 库中的 BPE tokenizer；该库用 Rust 实现核心算法以提升计算性能
- 我在 [./bytepair_encoder](../02_bonus_bytepair-encoder) 中创建了一个 Notebook，用来并排比较这两种实现（在示例文本上，tiktoken 大约快 5 倍）

In [25]:
# pip install tiktoken

In [37]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.13.0


In [38]:
tokenizer = tiktoken.get_encoding("gpt2")

In [39]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [40]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


- BPE tokenizer 会把未知词拆分成子词和单个字符：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/11.webp" width="1000px">

## 2.6 使用滑动窗口进行数据采样

- 我们训练 LLM 一次生成一个词，因此要相应地准备训练数据：序列中的下一个词就是要预测的目标：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="1000px">

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)#
print(len(enc_text))

5145


- 对每个文本片段，我们需要输入和目标
- 因为希望模型预测下一个词，所以目标就是输入整体向右移动一位后的结果

In [42]:
enc_sample = enc_text[50:]

print(enc_sample)

[290, 4920, 2241, 287, 257, 4489, 64, 319, 262, 34686, 41976, 13, 357, 10915, 314, 2138, 1807, 340, 561, 423, 587, 10598, 393, 28537, 2014, 198, 198, 1, 464, 6001, 286, 465, 13476, 1, 438, 5562, 373, 644, 262, 1466, 1444, 340, 13, 314, 460, 3285, 9074, 13, 46606, 536, 5469, 438, 14363, 938, 4842, 1650, 353, 438, 2934, 489, 3255, 465, 48422, 540, 450, 67, 3299, 13, 366, 5189, 1781, 340, 338, 1016, 284, 3758, 262, 1988, 286, 616, 4286, 705, 1014, 510, 26, 475, 314, 836, 470, 892, 286, 326, 11, 1770, 13, 8759, 2763, 438, 1169, 2994, 284, 943, 17034, 318, 477, 314, 892, 286, 526, 383, 1573, 11, 319, 9074, 13, 536, 5469, 338, 11914, 11, 33096, 663, 4808, 3808, 62, 355, 996, 484, 547, 12548, 287, 281, 13079, 410, 12523, 286, 22353, 13, 843, 340, 373, 407, 691, 262, 9074, 13, 536, 48819, 508, 25722, 276, 13, 11161, 407, 262, 40123, 18113, 544, 9325, 701, 11, 379, 262, 938, 402, 1617, 261, 12917, 905, 11, 5025, 502, 878, 402, 271, 10899, 338, 366, 31640, 12, 67, 20811, 1, 284, 910, 11, 351, 10

In [43]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


- 逐步来看，预测过程如下：

In [ ]:
for i in range(1, context_size+1):# 从 1 到 context_size（包含 context_size）进行迭代
    context = enc_sample[:i]
    desired = enc_sample[i]# 目标词

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [45]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


- 下一词预测会在后续章节介绍注意力机制之后再处理
- 现在先实现一个简单的数据加载器，它遍历输入数据集，并返回错开一位的输入和目标

- 安装并导入 PyTorch（安装建议见附录 A）

In [46]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.12.0+cpu


- 我们使用滑动窗口方法，每次将位置移动 +1：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/13.webp?123" width="1000px">

- 创建 dataset 和 dataloader，用于从输入文本数据集中提取片段

In [47]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):#stride 是滑动窗口的步长，默认值为 1
        self.input_ids = []
        self.target_ids = []

        # 对整段文本进行分词
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # 使用滑动窗口把文本切分成长度为 max_length 的重叠序列
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [50]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # 初始化 tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # 创建 dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # 创建 dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,# 是否在每个 epoch 开始时打乱数据
        drop_last=drop_last,# 是否在最后一个 batch 中丢弃剩余的样本
        num_workers=num_workers# 额外进程
    )

    return dataloader

- 以 batch size 为 1、上下文大小为 4 的 LLM 为例，测试 dataloader：

In [51]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [ ]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


- 下面的例子中，stride 等于上下文长度（这里是 4）：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="1000px">

- 也可以创建批量输出
- 注意这里增大了 stride，这样 batch 之间不会重叠，因为更多重叠可能导致过拟合加剧

In [54]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


&nbsp;
## 2.7 创建 token 嵌入

- 数据几乎已经可以供 LLM 使用了
- 最后一步是使用嵌入层，把 token 嵌入到连续向量表示中
- 通常，这些嵌入层本身也是 LLM 的一部分，并会在模型训练期间被更新（训练）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/15.webp" width="1000px">

- 假设有以下 4 个输入样本，它们的输入 ID 分别是 2、3、5 和 1（分词之后）：

In [55]:
input_ids = torch.tensor([2, 3, 5, 1])

- 为了简单起见，假设词表中只有 6 个词，我们想创建维度为 3 的嵌入：

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)#固定随机种子，便于教学和调试。
#Embedding 的权重一开始是随机初始化的
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- 这会得到一个 6x3 的权重矩阵：
- ID 0 -> 第 0 行向量
- ID 1 -> 第 1 行向量
- ......

In [57]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


- 对熟悉 one-hot 编码的人来说，上面的嵌入层方法本质上只是更高效地实现了 one-hot 编码后接全连接层矩阵乘法；补充代码 [./embedding_vs_matmul](../03_bonus_embedding-vs-matmul) 对此有说明
- 因为嵌入层只是 one-hot 编码加矩阵乘法的等价高效实现，所以它可以看作一个能够通过反向传播优化的神经网络层

- 要把 ID 为 3 的 token 转换成 3 维向量，可以这样做：

In [58]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


- 注意，上面的结果是 `embedding_layer` 权重矩阵中的第 4 行
- 要嵌入上面全部 4 个 `input_ids` 值，可以这样做

In [46]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


- 嵌入层本质上是一次查表操作：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/16.webp?123" width="1000px">

- **你可能会对比较嵌入层和普通线性层的 bonus 内容感兴趣：[../03_bonus_embedding-vs-matmul](../03_bonus_embedding-vs-matmul)**

&nbsp;
## 2.8 编码词的位置

- 无论 ID 位于输入序列中的哪个位置，嵌入层都会把它们转换成相同的向量表示：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/17.webp" width="1000px">

- 位置嵌入会与 token 嵌入向量组合，形成大语言模型的输入嵌入：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/18.webp" width="1000px">

- BytePair 编码器的词表大小为 50,257：
- 假设我们想把输入 token 编码成 256 维向量表示：

In [59]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- 从 dataloader 中采样数据时，我们会把每个 batch 中的 token 嵌入成 256 维向量
- 如果 batch size 为 8，每个样本有 4 个 token，结果就是一个 8 x 4 x 256 的张量：

In [ ]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)#iter() 函数用于创建一个迭代器对象。我们用它来从 dataloader 中逐批获取数据
inputs, targets = next(data_iter)

In [61]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [ ]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# 嵌入的具体样子
print(token_embeddings)

torch.Size([8, 4, 256])
tensor([[[ 0.4913,  1.1239,  1.4588,  ..., -0.3995, -1.8735, -0.1445],
         [ 0.4481,  0.2536, -0.2655,  ...,  0.4997, -1.1991, -1.1844],
         [-0.2507, -0.0546,  0.6687,  ...,  0.9618,  2.3737, -0.0528],
         [ 0.9457,  0.8657,  1.6191,  ..., -0.4544, -0.7460,  0.3483]],

        [[ 1.5460,  1.7368, -0.7848,  ..., -0.1004,  0.8584, -0.3421],
         [-1.8622, -0.1914, -0.3812,  ...,  1.1220, -0.3496,  0.6091],
         [ 1.9847, -0.6483, -0.1415,  ..., -0.3841, -0.9355,  1.4478],
         [ 0.9647,  1.2974, -1.6207,  ...,  1.1463,  1.5797,  0.3969]],

        [[-0.7713,  0.6572,  0.1663,  ..., -0.8044,  0.0542,  0.7426],
         [ 0.8046,  0.5047,  1.2922,  ...,  1.4648,  0.4097,  0.3205],
         [ 0.0795, -1.7636,  0.5750,  ...,  2.1823,  1.8231, -0.3635],
         [ 0.4267, -0.0647,  0.5686,  ..., -0.5209,  1.3065,  0.8473]],

        ...,

        [[-1.6156,  0.9610, -2.6437,  ..., -0.9645,  1.0888,  1.6383],
         [-0.3985, -0.9235, -1.31

- GPT-2 使用绝对位置嵌入，所以这里只需要再创建一个嵌入层：

In [ ]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

#嵌入层权重的具体样子
print(pos_embedding_layer.weight)

Parameter containing:
tensor([[-0.8290,  0.0647,  0.1265,  ...,  0.3610,  1.0378,  0.7732],
        [-0.3407, -0.0392,  1.8435,  ...,  0.5663,  1.3551, -1.9100],
        [-1.0489, -0.4816, -0.8424,  ...,  0.9912, -0.6396,  1.3296],
        [ 0.2580,  0.3851,  0.0491,  ...,  1.3488, -1.6770, -0.4796]],
       requires_grad=True)


In [ ]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))#arange 函数返回一个一维张量，包含从 0 到 max_length-1 的整数。我们用它来获取每个位置的嵌入
print(pos_embeddings.shape)

#嵌入的具体样子
print(pos_embeddings)

torch.Size([4, 256])
tensor([[-0.8290,  0.0647,  0.1265,  ...,  0.3610,  1.0378,  0.7732],
        [-0.3407, -0.0392,  1.8435,  ...,  0.5663,  1.3551, -1.9100],
        [-1.0489, -0.4816, -0.8424,  ...,  0.9912, -0.6396,  1.3296],
        [ 0.2580,  0.3851,  0.0491,  ...,  1.3488, -1.6770, -0.4796]],
       grad_fn=<EmbeddingBackward0>)


- 要创建 LLM 使用的输入嵌入，只需把 token 嵌入和位置嵌入相加：
- PyTorch 会把 pos_embeddings 自动看成：

[1, 4, 256]
然后复制到 8 个 batch 上：

[8, 4, 256]

In [71]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# 取入的具体样子
print(input_embeddings)

torch.Size([8, 4, 256])
tensor([[[-0.3377,  1.1886,  1.5854,  ..., -0.0385, -0.8358,  0.6286],
         [ 0.1073,  0.2144,  1.5780,  ...,  1.0660,  0.1560, -3.0944],
         [-1.2996, -0.5362, -0.1737,  ...,  1.9530,  1.7341,  1.2767],
         [ 1.2037,  1.2509,  1.6682,  ...,  0.8944, -2.4230, -0.1313]],

        [[ 0.7170,  1.8016, -0.6582,  ...,  0.2606,  1.8962,  0.4311],
         [-2.2029, -0.2306,  1.4623,  ...,  1.6883,  1.0056, -1.3009],
         [ 0.9358, -1.1299, -0.9839,  ...,  0.6071, -1.5751,  2.7774],
         [ 1.2227,  1.6825, -1.5716,  ...,  2.4951, -0.0973, -0.0827]],

        [[-1.6003,  0.7219,  0.2928,  ..., -0.4433,  1.0919,  1.5158],
         [ 0.4638,  0.4655,  3.1357,  ...,  2.0311,  1.7648, -1.5894],
         [-0.9695, -2.2452, -0.2674,  ...,  3.1735,  1.1835,  0.9661],
         [ 0.6847,  0.3204,  0.6177,  ...,  0.8279, -0.3705,  0.3677]],

        ...,

        [[-2.4446,  1.0257, -2.5171,  ..., -0.6034,  2.1266,  2.4115],
         [-0.7392, -0.9627,  0.52

- 在输入处理流程的初始阶段，输入文本会被切分成独立的 token
- 随后，根据预定义词表把这些 token 转换为 token ID：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/19.webp" width="1000px">

&nbsp;
## 小结与要点

参见 [./dataloader.ipynb](./dataloader.ipynb) 代码 Notebook；它是本章实现的数据加载器的精简版本，后续章节训练 GPT 模型时会用到。

练习答案见 [./exercise-solutions.ipynb](./exercise-solutions.ipynb)。

如果你想了解如何从零实现并训练 GPT-2 tokenizer，可以查看 [Byte Pair Encoding (BPE) Tokenizer From Scratch](../02_bonus_bytepair-encoder/compare-bpe-tiktoken.ipynb) Notebook。